# 01: EEGBCI sanity check

**Feasibility question 1:** does the pipeline measure real brain rhythms, not artifacts of our own processing?

**Test:** EEGBCI baseline runs. R01 = eyes open, R02 = eyes closed. Closing the eyes is a large, well-established increase in alpha (8–13 Hz), strongest over occipital/parietal sites. If our pipeline can't see that, it can't be trusted for anything subtler.

**Pass criteria (set before running, also recorded in Form B2):**
1. Relative alpha at Oz is higher eyes-closed than eyes-open in ≥ 90% of subjects
2. Paired effect size (Cohen's d_z) on log alpha power at Oz ≥ 0.8
3. Cleaning keeps ≥ 80% of windows on average (rejection isn't throwing the data away)

This notebook only *uses* `preproc.py`; all processing logic lives there.

In [20]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

import preproc

warnings.filterwarnings("ignore", category=RuntimeWarning)
mne.set_log_level("ERROR")
%matplotlib inline

# ---- settings for this analysis ----
SUBJECTS = list(range(1, 5))                     # quick first pass; EEGBCI has 109 (use range(1, 11) for more)
RUNS = {1: "eyes_open", 2: "eyes_closed"}
QC_SUBJECT = 1                       # subject shown in the quality-check plots
CHANNEL = "Oz"                       # where the eyes-closed alpha effect is strongest

PASS = {"pct_subjects": 90, "min_dz": 0.8, "min_kept_pct": 80}

s = preproc.get_settings(montage='standard_1020')          # defaults; override here if needed, e.g. win_s=2
OUT = Path("results/01_eegbci"); OUT.mkdir(parents=True, exist_ok=True)
s

{'montage': 'standard_1020',
 'resample_hz': 160.0,
 'l_freq': 1.0,
 'h_freq': 40.0,
 'notch_hz': 60.0,
 'bad_z_thresh': 5.0,
 'flat_uv': 1.0,
 'reference': 'average',
 'use_ica': True,
 'ica_n_components': 0.99,
 'eog_proxy': ['Fp1', 'Fp2'],
 'reject_ptp_uv': 'auto',
 'reject_auto_k': 5.0,
 'win_s': 4.0,
 'step_s': 2.0,
 'bands': {'theta': (4.0, 8.0), 'alpha': (8.0, 13.0), 'beta': (13.0, 30.0)},
 'total_band': (1.0, 40.0),
 'ratio_num': ('theta', 'Fz'),
 'ratio_den': ('alpha', 'Pz'),
 'channels': None,
 'random_state': 42}

## 1. Run the pipeline
Each subject × run goes through `preproc.run_pipeline`. A failure on one subject is logged and skipped rather than stopping the whole loop.

In [21]:
feats, logs, qc = [], [], {}

for subj in SUBJECTS:
    for run, cond in RUNS.items():
        try:
            raw = preproc.load_eegbci(subj, run)
            clean, f, log = preproc.run_pipeline(raw, s, subject=subj, condition=cond)
            feats.append(f)
            logs.append(log)
            if subj == QC_SUBJECT:
                qc[cond] = {"raw": raw, "clean": clean}
        except Exception as e:
            print(f"S{subj:03d} {cond}: FAILED ({e})")
            logs.append({"subject": subj, "condition": cond, "error": str(e)})

F = pd.concat(feats, ignore_index=True)
L = pd.DataFrame(logs)
F.to_csv(OUT / "features.csv", index=False)
L.to_csv(OUT / "pipeline_log.csv", index=False)
print(f"{F.subject.nunique()} subjects, {len(F)} feature rows saved to {OUT}")

4 subjects, 11776 feature rows saved to results/01_eegbci


## 2. Quality checks
Look at these before trusting any result. Things to watch for: many bad channels in one subject, ICA removing lots of components, or a high share of dropped windows.

In [22]:
L["kept_pct"] = 100 * L["windows_kept"] / (L["windows_kept"] + L["windows_dropped"])
L["n_bad_channels"] = L["bad_channels"].apply(lambda b: len(b) if isinstance(b, list) else np.nan)
L["n_ica_removed"] = L["ica_excluded"].apply(lambda b: len(b) if isinstance(b, list) else np.nan)
L[["subject", "condition", "bad_channels", "n_ica_removed", "windows_kept", "windows_dropped", "kept_pct"]]

,subject,condition,bad_channels,n_ica_removed,windows_kept,windows_dropped,kept_pct
0,1,eyes_open,[T10],2,24,5,82.758621
1,1,eyes_closed,[T10],1,26,3,89.655172
2,2,eyes_open,[],2,21,8,72.413793
3,2,eyes_closed,[],1,29,0,100.000000
4,3,eyes_open,[CP6],3,26,3,89.655172
5,3,eyes_closed,"[C4, CP6, T10]",1,26,3,89.655172
6,4,eyes_open,[FT8],2,21,8,72.413793
7,4,eyes_closed,[],1,11,18,37.931034


In [23]:
# Raw vs cleaned overlay: Fp1 shows blink removal, Oz shows alpha is preserved
cond = "eyes_open"
raw_view = preproc.filter_raw(preproc.standardize(qc[cond]["raw"], s), s)  # same filter, no cleaning
clean = qc[cond]["clean"]
t0, dur = 20, 10                                                           # seconds
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
for ax, ch in zip(axes, ["Fp1", "Oz"]):
    a = raw_view.copy().crop(t0, t0 + dur)
    b = clean.copy().crop(t0, t0 + dur)
    ax.plot(a.times + t0, a.get_data(picks=ch)[0] * 1e6, lw=0.8, label="filtered only", color="0.6")
    ax.plot(b.times + t0, b.get_data(picks=ch)[0] * 1e6, lw=0.8, label="cleaned", color="C0")
    ax.set_ylabel(f"{ch} (µV)")
axes[0].legend(loc="upper right")
axes[0].set_title(f"S{QC_SUBJECT:03d} {cond}: before vs after bad-channel fix, re-reference and ICA")
axes[1].set_xlabel("time (s)")
fig.tight_layout(); fig.savefig(OUT / "qc_overlay.png", dpi=150)

In [24]:
# Spectrum at Oz, eyes open vs closed (QC subject): expect a clear 8-13 Hz peak eyes-closed
fig, ax = plt.subplots(figsize=(7, 4))
for cond, color in [("eyes_open", "C0"), ("eyes_closed", "C3")]:
    spec = qc[cond]["clean"].compute_psd(method="welch", fmin=1, fmax=40, picks=[CHANNEL], n_fft=256)
    p, f = spec.get_data(return_freqs=True)
    ax.semilogy(f, p[0] * 1e12, label=cond, color=color)
ax.axvspan(8, 13, color="0.9", zorder=0, label="alpha band")
ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("power (µV²/Hz)")
ax.set_title(f"S{QC_SUBJECT:03d}, {CHANNEL}"); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "qc_psd.png", dpi=150)

## 3. Result
One number per subject per condition: the median across windows at the chosen channel. Then compare eyes-closed vs eyes-open within each subject.

In [25]:
per_subj = (
    F[F.channel == CHANNEL]
    .groupby(["subject", "condition"])[["alpha_rel", "log_alpha_abs"]]
    .median()
    .unstack("condition")
    .dropna()
)
diff_rel = per_subj["alpha_rel"]["eyes_closed"] - per_subj["alpha_rel"]["eyes_open"]
diff_log = per_subj["log_alpha_abs"]["eyes_closed"] - per_subj["log_alpha_abs"]["eyes_open"]

n = len(per_subj)
pct_up = 100 * (diff_rel > 0).mean()
dz = diff_log.mean() / diff_log.std(ddof=1) if n > 1 else np.nan  # paired effect size
kept = L["kept_pct"].mean()

summary = pd.DataFrame({
    "measure": ["subjects with alpha increase (%)", "Cohen's d_z (log alpha)", "windows kept (%)"],
    "result": [round(pct_up, 1), round(dz, 2), round(kept, 1)],
    "target": [f">= {PASS['pct_subjects']}", f">= {PASS['min_dz']}", f">= {PASS['min_kept_pct']}"],
    "pass": [pct_up >= PASS["pct_subjects"], dz >= PASS["min_dz"], kept >= PASS["min_kept_pct"]],
})
if n >= 6:
    from scipy.stats import wilcoxon
    print(f"Wilcoxon signed-rank p = {wilcoxon(diff_log).pvalue:.4g} (n = {n})")
else:
    print(f"n = {n}: too few subjects for a meaningful test or effect size; add subjects.")
summary.to_csv(OUT / "summary.csv", index=False)
summary

n = 4: too few subjects for a meaningful test or effect size; add subjects.


,measure,result,target,pass
0,subjects with alpha increase (%),100.00,>= 90,True
1,Cohen's d_z (log alpha),4.25,>= 0.8,True
2,windows kept (%),79.30,>= 80,False


In [26]:
# Within-subject view (meaningful with 1 subject): windows are the repeated measures
from scipy.stats import mannwhitneyu

subj = QC_SUBJECT
colors = {"eyes_open": "C0", "eyes_closed": "C3"}
Fs = F[F.subject == subj]
Fo = Fs[Fs.channel == CHANNEL]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3), gridspec_kw={"width_ratios": [1, 1.5, 1.2]})

# (a) window-level relative alpha at Oz: distribution per condition
ax = axes[0]
vals = [Fo[Fo.condition == c]["alpha_rel"].values for c in colors]
bp = ax.boxplot(vals, positions=[0, 1], widths=0.5, showfliers=False, patch_artist=True)
for patch, c in zip(bp["boxes"], colors):
    patch.set(facecolor=colors[c], alpha=0.3)
for med in bp["medians"]:
    med.set_color("k")
rng = np.random.default_rng(0)
for i, (c, v) in enumerate(zip(colors, vals)):
    ax.scatter(i + rng.uniform(-0.12, 0.12, len(v)), v, s=18, color=colors[c], alpha=0.8, zorder=3)
p = mannwhitneyu(vals[1], vals[0], alternative="greater").pvalue
ax.set_xticks([0, 1], ["eyes open", "eyes closed"])
ax.set_ylabel(f"relative alpha at {CHANNEL}")
ax.set_title(f"Per-window alpha (n={len(vals[0])} / {len(vals[1])})\nMann-Whitney p = {p:.2g}")

# (b) alpha over time within each run
ax = axes[1]
for c in colors:
    d = Fo[Fo.condition == c]
    ax.plot(d["t_start_s"], d["alpha_rel"], marker="o", ms=4, lw=1, color=colors[c], label=c.replace("_", " "))
ax.set_xlabel("window start (s, within run)"); ax.set_ylabel(f"relative alpha at {CHANNEL}")
ax.set_title("Stability across the recording"); ax.legend()

# (c) scalp map of the eyes-closed minus eyes-open change in log alpha power
ax = axes[2]
m = (Fs.groupby(["channel", "condition"])["log_alpha_abs"].median().unstack("condition"))
diff = (m["eyes_closed"] - m["eyes_open"])
info = qc["eyes_closed"]["clean"].info
diff = diff.reindex(info.ch_names)
good = diff.notna().values
info_g = mne.pick_info(info, np.where(good)[0])
vmax = np.nanmax(np.abs(diff.values))
im, _ = mne.viz.plot_topomap(diff.values[good], info_g, axes=ax, show=False, cmap="RdBu_r", vlim=(-vmax, vmax))
fig.colorbar(im, ax=ax, shrink=0.8, label="Δ log10 alpha power (closed − open)")
ax.set_title("Where alpha rises (expect occipital)")

fig.suptitle(f"S{subj:03d}: eyes-closed vs eyes-open alpha", y=1.02)
fig.tight_layout(); fig.savefig(OUT / "result_within_subject.png", dpi=150, bbox_inches="tight")

# Across-subject paired plot, only when there is more than one subject
if n > 1:
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    for _, row in per_subj["alpha_rel"].iterrows():
        up = row["eyes_closed"] > row["eyes_open"]
        ax.plot([0, 1], [row["eyes_open"], row["eyes_closed"]], marker="o",
                color="C0" if up else "C3", alpha=0.8)
    ax.set_xticks([0, 1], ["eyes open", "eyes closed"])
    ax.set_ylabel(f"relative alpha power at {CHANNEL}")
    ax.set_title(f"{int((diff_rel > 0).sum())}/{n} subjects increase")
    fig.tight_layout(); fig.savefig(OUT / "result_paired.png", dpi=150)